In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw=pd.read_csv("../data/raw/ecommerce_sales_data.csv",dtype=str,keep_default_na=False)

In [3]:
df_clean = df_raw.copy(deep=True)

In [4]:
df_clean.shape

(1031, 13)

In [5]:
df_clean=df_clean[~df_clean.apply(
        lambda row: row.str.strip().eq("").all(),
        axis=1
    )
]

In [6]:
df_clean = df_clean.dropna(how='all')

In [7]:
df_clean.columns=[col.strip().lower().replace(" ", "_").replace("?", "") for col in df_clean.columns]

In [8]:
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first')
rows_removed = rows_before - len(df_clean)
print(rows_removed)

18


In [9]:
df_clean['missing_count'] = df_clean.isna().sum(axis=1)
df_clean = df_clean.sort_values('missing_count')

In [10]:
df_clean = df_clean.drop_duplicates(
    subset=['order_id'],
    keep='first'
)


In [21]:
df_clean=df_clean.reset_index(drop=True)
df_clean

,order_id,customer_id,order_date,category,product_name,qty,unit_price,country,paymentmethod,customer_email,rating,discount,returned,missing_count
0,ORD1087,CUST157,2023/03/03,Clothing,Bluetooth Speaker,9.0,NaN,USA,Upi,customer45@mail.com,3.0,0.30,No,0
1,ORD1871,CUST266,2024-09-16,Home & Kitchen,T-Shirt,10.0,NaN,Australia,Credit Card,customer41@mail.com,2.0,0.48,Yes,0
2,ORD1130,CUST45,20-Feb-2023,Books,Face Cream,6.0,NaN,Unknown,,customer280@mail.com,2.0,0.40,No,0
3,ORD1661,,30/07/2024,Electronics,Notebook,9.0,NaN,France,Upi,customer11@mail.com,4.0,0.09,No,0
4,ORD1308,CUST3,2024-05-27,Sports,USB-C Cable,NaN,NaN,Unknown,Upi,customer232@mail.com,1.0,0.43,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
961,ORD1855,CUST72,10-12-2024,Electronics,,5.0,NaN,Unknown,Cod,customer75@mail.com,5.0,0.34,,0
962,ORD1466,CUST76,27.03.2024,Home & Kitchen,Headphones,5.0,NaN,Australia,Credit Card,customer71@mail.com,4.0,0.33,No,0
963,ORD1121,CUST208,2023-06-28,Electronics,,8.0,NaN,Unknown,Credit Card,,3.0,0.39,No,0
964,ORD1860,CUST78,12-Mar-2024,Home & Kitchen,wireless mouse,1.0,NaN,USA,Credit Card,customer263@mail.com,1.0,0.46,Yes,0


In [12]:
df_clean['customer_id'] = df_clean['customer_id'].str.upper()
df_clean['customer_email'] = df_clean['customer_email'].str.lower().str.replace('@@', '@', regex=False)

In [13]:
country_map = {
      'USA': 'USA',
      'U.S.A': 'USA',
      'United States': 'USA',
      'India': 'India',
      'india': 'India',
      'INDIA': 'India',
      'UK': 'UK',
      'United Kingdom': 'UK',
      'Germany': 'Germany',
      'germany': 'Germany',
      'France': 'France',
      'france': 'France',
      'Australia': 'Australia',
      'N/A': 'Unknown',
      'unknown': 'Unknown',
      '': 'Unknown',
  }

df_clean["country"]=df_clean["country"].replace(country_map)

In [14]:
df_clean['category'] = df_clean['category'].str.title()

df_clean['paymentmethod'] = df_clean['paymentmethod'].str.title()

df_clean['category'] = df_clean['category'].replace({
    'Home And Kitchen': 'Home & Kitchen'
})

In [15]:
returned_map = {
    'Yes': 'Yes',
    'yes': 'Yes',
    'Y': 'Yes',
    'True': 'Yes',
    '1': 'Yes',
    'No': 'No',
    'no': 'No',
    'N': 'No',
    'False': 'No',
    '0': 'No',
}

df_clean['returned'] = df_clean['returned'].replace(returned_map)

In [16]:
df_clean['qty'] = df_clean['qty'].replace({'five': '5', 'nine': '9'})
df_clean['qty'] = pd.to_numeric(df_clean['qty'], errors='coerce')
df_clean.loc[
    (df_clean['qty'] <= 0) | (df_clean['qty'] > 100),
    'qty'
] = np.nan


In [17]:
df_clean['unit_price'] = (
    df_clean['unit_price']
    .astype(str)
    .str.replace(r'[\$,Rs.]', '', regex=True)
)

df_clean['unit_price'] = df_clean['unit_price'].replace({
    'free': '0.0'
})

df_clean['unit_price'] = pd.to_numeric(
    df_clean['unit_price'],
    errors='coerce'
)

df_clean.loc[
    (df_clean['unit_price'] <= 0) |
    (df_clean['unit_price'] > 1000),
    'unit_price'
] = np.nan

In [18]:
df_clean['rating'] = df_clean['rating'].replace({
    'good': '4.0'
})

df_clean['rating'] = pd.to_numeric(
    df_clean['rating'],
    errors='coerce'
)

df_clean.loc[
    (df_clean['rating'] < 1) |
    (df_clean['rating'] > 5),
    'rating'
] = np.nan

In [19]:
def clean_discount(value):
    value = str(value).strip()
    if value == '':
        return np.nan

    if '%' in value:
        value = value.replace('%', '')
        
        return float(value) / 100

    return float(value)


df_clean['discount'] = df_clean['discount'].apply(
    lambda x: clean_discount(x)
)

df_clean.loc[
    (df_clean['discount'] < 0) |
    (df_clean['discount'] > 1),
    'discount'
] = np.nan

In [20]:
print(
    df_clean[
        ['qty', 'unit_price', 'rating', 'discount']
    ].describe()
)

              qty  unit_price      rating    discount
count  862.000000   54.000000  861.000000  855.000000
mean     5.504640  575.722222    2.954704    0.240690
std      2.866256  265.183839    1.405650    0.143451
min      1.000000   69.000000    1.000000    0.000000
25%      3.000000  377.000000    2.000000    0.120000
50%      5.000000  600.000000    3.000000    0.230000
75%      8.000000  769.000000    4.000000    0.360000
max     10.000000  999.000000    5.000000    0.500000


In [22]:
df_clean['order_date'] = (
    df_clean['order_date']
    .astype(str)
    .str.strip()
)

df_clean['order_date'] = pd.to_datetime(
    df_clean['order_date'],
    errors='coerce',
    dayfirst=True,
    format='mixed'
)

In [23]:
print(
    df_clean['order_date']
    .dt.year
    .value_counts(dropna=False)
)

order_date
2024.0    471
2023.0    463
NaN        32
Name: count, dtype: int64


In [24]:
product_category_rules = {
    'T-Shirt': 'Clothing',
    'Denim Jacket': 'Clothing',
    'Shampoo': 'Beauty',
    'Face Cream': 'Beauty',
    'Gel Pen': 'Books',
    'Notebook': 'Books',
    'Yoga Mat': 'Sports',
    'Running Shoes': 'Sports',
    'Laptop Stand': 'Electronics',
    'Headphones': 'Electronics',
    'Bluetooth Speaker': 'Electronics',
    'Wireless Mouse': 'Electronics',
    'USB-C Cable': 'Electronics',
    'Webcam HD': 'Electronics',
    'Power Bank': 'Electronics',
    'Smart Watch': 'Electronics',
    'Phone Case': 'Electronics',
}

df_clean['category'] = df_clean['product_name'].map(
    product_category_rules
)